# WS2 — The Pitch Grammar: A Bayesian Variable-Order Markov Model

**Workstream 2 of the pitch-sequencing rigor ladder.** This notebook fits and reads the
study's *grammar* rung: a per-context **variable-order Markov model** of next-pitch
**selection**, built from hierarchical Dirichlet backoff over ordered pitch-family tokens.
Where WS1 asked "can a shrunk lookup table see order?" and answered with a flat, fixed key,
WS2 asks the sharper question — *how deep does the ordered dependence actually go, and which
ordered rules does the data support?* — and lets the depth of context be **earned** by the
data, one level at a time.

## The metaphor, and what this rung measures

Treat each plate appearance as a **sentence** and each pitch family as a **word**. A pitcher's
selection is then a little grammar: some word sequences are common, some are avoided, and the
question is whether knowing the *ordered* words so far sharpens the guess for the next word
beyond just the last word (a bigram) or the count and the pitcher (no history at all). WS2 is
the **finding #1 instrument** — it speaks to *selection structure* (does prior order help
predict what is thrown next?) and is deliberately silent about *outcome* value. That firewall
is load-bearing and we restate it throughout.

The project keeps three findings separate that the applied literature routinely conflates
(SPEC §0, verbatim):

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the
>    current pitch, after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.
>
> The first is easy; the second is hard; the third needs assumptions that public data
> cannot fully satisfy.

WS2 lives squarely in **finding #1**. Its headline instruments are the ordered-selection edge
`delta_order_L1 = Loss(L1) − Loss(O)` (does order beat the previous pitch?), the
**effective-order** distribution (how deep does the data actually let us go?), the
**bits-of-predictability** `B_seq` (SPEC §10), and the **grammar exhibits** (the ordered
"rules", ranked by how much they move the next-token distribution).

## The DATA_MODE toggle — and the D30 inversion (read this first)

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a
single toggle, `DATA_MODE`, selects the world:

- `'synth_null'` — the correctness oracle's **null world**. **Default.** This is where WS2
  differs from every other workstream: *the null world is WS2's **positive control**.*
- `'synth_positive'` — the oracle's positive world (a planted ordered *outcome* effect on
  whiffs), shown for completeness only.
- `'real'` — the real decision table built by `python -m pitchseq.build_table` (Phase 2).

**Why the null world is a positive control here (decision D30).** The oracle's worlds were
designed (decision D20) to *separate finding #1 from finding #2*: the "null" world forbids an
ordered **outcome** effect, but it deliberately **plants an ordered *selection* habit** — a
mild *no-three-in-a-row* tendency (a pitcher who just threw two of the same family is less
likely to throw a third). For an *outcome* model (WS1's run-value tables, WS3's outcome stack)
that world is a true null: nothing to find. But WS2 is a **selection** model, and the planted
selection habit is exactly an order-2 grammar rule. So for WS2 the null world is the world
*with* a signal — a **positive control**: the grammar must **detect** effective order ≥ 2 and
the repeat-suppression motif. WS2's *negative* control is a different operation entirely — a
**stratified history permutation** that scrambles the ordered contexts, under which the
grammar's edge and effective order must **collapse**. Keep this inversion in mind: on the
`synth_null` default, *seeing order is the correct answer.*

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will
happen and why; **after** each cell we say how to read what came out. Numbers that depend on
the real data are `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply appear when
you run the cell. The **Results** section (§9) is *branched* on three axes (Grammar × Bits ×
Motif): a code cell inspects the computed numbers and prints which branch applies, and the
markdown that follows holds the pre-written interpretation for every branch, so the conclusion
is ready the instant the cells finish. The exact formulas live in `THEORY.md`; the exact code
in `workstreams/ws2_bayes_markov/model.py`; the plain-English tour in `SEAN-README.md`.

## 1. Setup

We put the repository root on `sys.path` (so the `pitchseq` package and the
`workstreams.ws2_bayes_markov` module import whether the notebook is launched from the repo
root or from `notebooks/`), load the shared study config, and set the knobs that steer the
whole notebook: `DATA_MODE` (which world), `SEED` (reproducibility), `K_MAX` (the grammar's
order cap), and `VIEWS`. Nothing here touches data yet.

In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits
from pitchseq.families import FAMILIES
from pitchseq.eval.baselines import build_baselines
from pitchseq.eval.predictions import ACTION_PROB_COLS
from pitchseq.eval.metrics import reliability_table, log_loss_per_row, clustered_ci
from pitchseq.eval.harness import evaluate_predictions, slice_masks
from pitchseq.eval.predictability import bits_of_predictability, bits_summary

# --- WS2 (this workstream) ---
from workstreams.ws2_bayes_markov import model as vom
from workstreams.ws2_bayes_markov.run_ws2 import run_ws2

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_null' | 'synth_positive' | 'real'.
# NOTE (D30): on 'synth_null' the planted order-2 selection habit is REAL signal — the null
# world is WS2's POSITIVE control. Seeing order there is the correct answer.
DATA_MODE = "synth_null"

# The feasible views for a Markov grammar (D29). U and OM are NOT-APPLICABLE:
#   U  — a Markov context is an ordered string; an unordered multiset is not a Markov context.
#   OM — the grammar is a within-PA chain; matchup memory is cross-PA.
# The run prints why rather than fitting them (see the not_applicable_note calls below).
VIEWS = ["C", "L1", "O"]

# Grammar-order cap for the O view (decision D29; C is order 0, L1 is order 1).
K_MAX = vom.K_MAX_DEFAULT  # 4

# Cluster-bootstrap replicates for the confidence intervals. Only affects CI width; lower it
# for a quick pass on the full validation season (SPEC §7 clusters by pitcher-game).
N_BOOT = 100

# Where the real decision table lives after `python -m pitchseq.build_table` (Phase 2).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"views     : {VIEWS}   (U/OM not-applicable for a Markov grammar, D29)")
print(f"K_MAX     : {K_MAX}")
print(f"seed      : {SEED}")

### Plotting style (fixed, colorblind-safe view colours)

Every figure uses the **Okabe–Ito** palette (safe for the common forms of colour-vision
deficiency) and a **fixed view → colour map**, identical to WS1's notebook, so a given state
view is *always* the same colour throughout the study: **C** is blue, **L1** bluish-green,
**O** vermillion. We strip the top and right spines and drop gridlines by default. This cell
defines the palette, the map, and two small helpers used everywhere below.

In [ ]:
# Okabe–Ito qualitative palette (colorblind-safe).
OKABE_ITO = {
    "orange":        "#E69F00",
    "sky_blue":      "#56B4E9",
    "bluish_green":  "#009E73",
    "yellow":        "#F0E442",
    "blue":          "#0072B2",
    "vermillion":    "#D55E00",
    "reddish_purple":"#CC79A7",
    "black":         "#000000",
}

# Fixed view -> colour, IDENTICAL to WS1. C is the context-only base; O is the fully ordered
# headline grammar. U/OM are kept for a consistent legend even though WS2 does not fit them.
VIEW_COLORS = {
    "C":  OKABE_ITO["blue"],
    "U":  OKABE_ITO["orange"],
    "L1": OKABE_ITO["bluish_green"],
    "O":  OKABE_ITO["vermillion"],
    "OM": OKABE_ITO["reddish_purple"],
}

# A neutral colour for the count-based reference baselines.
REF_COLOR = OKABE_ITO["black"]

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.autolayout": True,
})


def style_axes(ax):
    '''Left+bottom spines only; no top/right. Returns the axis for chaining.'''
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    '''One figure, one axis, pre-styled.'''
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax


# Family -> column index (FAMILIES order) and the "no prior pitch" sentinel used by the
# grammar's lag encoding (== len(FAMILIES); families are 0..7).
FAM_INDEX = {f: i for i, f in enumerate(FAMILIES)}
NONE_CODE = len(FAMILIES)


def family_codes(series) -> np.ndarray:
    '''Integer family code (0..7) per row, XX/unknown -> last index.'''
    obj = series.astype("object").to_numpy()
    return np.array([FAM_INDEX.get(f, len(FAMILIES) - 1) for f in obj], dtype=np.int64)

## 2. Data and exploratory analysis — the grammar's raw material

### Plate appearances as sentences, families as words

Every workstream reads the same **canonical decision table** (SPEC §3): one row per pitch,
the *decision made immediately before that pitch is released*, keyed by
`(game_pk, at_bat_number, pitch_number)`. For WS2 the relevant columns are minimal and shared:
`pa_id` (the sentence id), `pitch_number` (word position within the sentence), `family` (the
word — one of the 8 pitch families), and the conditioning coordinates the grammar keys on —
the count `(balls, strikes)`, handedness `(stand, p_throws)`, and `pitcher`. The grammar
**re-implements no history logic**: it forms ordered token contexts straight from `pa_id` /
`pitch_number` / `family`, and everything else (leakage-safe rolling features, the audited
column set) it inherits by reading only the shared table.

The **one non-negotiable discipline** is leakage (SPEC §0): a feature for the decision before
pitch *t* must be computable strictly *before pitch t is thrown*. The grammar's context is the
families of **prior** pitches this PA — all known before the current decision — so it is
leakage-safe by construction. The current pitch's own family is the *target*, never part of
the context.

The next cell loads or synthesises the table for `DATA_MODE`, then applies the locked
**temporal split** (SPEC §7): train on 2021–2023, validate on 2024. Splits are by season,
never by random row — pitches in the same PA/game are far too dependent for a random split to
be honest.

In [ ]:
def load_decision_table(mode: str, config: dict, seed: int):
    '''Return (decision_table, truth_meta) for the chosen world.'''
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. "
                "Build it first with `python -m pitchseq.build_table` (RUNBOOK Step 1)."
            )
        table = pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow")
        return table, {"world": "real"}

    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world

    # n_games=300 reproduces the committed WS2 validation scale (train~29.8k / val~9.9k),
    # large enough for the planted order-2 habit's depth-2 contexts to clear their support.
    if mode == "synth_null":
        raw, truth = make_null_world(n_games=300, seed=seed, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(
            n_games=300, seed=seed, innings_per_game=6,
            effect_size=0.30, velo_gap_threshold=5.0,
        )
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_decision_table(DATA_MODE, CONFIG, SEED)

splits = make_splits(table, CONFIG)["primary"]
train = table.loc[splits["train"].to_numpy()].reset_index(drop=True)
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)

print(f"world        : {truth.get('world', DATA_MODE)}")
print(f"total rows   : {len(table):,}")
print(f"train rows   : {len(train):,}   (seasons {CONFIG['split']['train']})")
print(f"val rows     : {len(val):,}   (seasons {CONFIG['split']['val']})")
print(f"columns      : {len(table.columns)}")
if DATA_MODE == "synth_null":
    print("NOTE (D30): this is WS2's POSITIVE control — a planted order-2 'no-three-in-a-row'")
    print("            selection habit. The grammar SHOULD detect effective order >= 2 here.")

**How to read the split.** On the synthetic worlds the season is assigned cyclically, so the
train/val split is populated even though the world is small. On real data `train` is ~2.3M
pitches and `val` ~0.77M. If either side is empty, the season column is wrong — stop and check
the build. The four EDA figures below (one per cell) are the grammar's raw material: the
transitions it will model, the n-grams it will score, the repeats it will suppress, and the
sentence lengths that decide how deep order can even reach.

### (a) The within-PA transition matrix — the bigram the grammar starts from

The crudest form of "does order matter for selection?" is the **first-order transition matrix**:
`P(next family | previous family)`, computed within plate appearances. Row = the previous
pitch's family, column = the next pitch's family, each **row sums to 1**. This is exactly what
the **L1** view captures. A flat matrix (every row identical) means the previous pitch tells
you nothing; structure — especially off the diagonal — is first-order selection dependence.
The **diagonal** is the repeat tendency, which the planted null-world habit deliberately
depresses.

In [ ]:
lag1 = vom.lag_family_codes(train, 1)[:, 0]      # previous-family code (NONE_CODE if no prior)
nxt = family_codes(train["family"])              # current (next) family code
has_prev = lag1 != NONE_CODE

T = np.zeros((len(FAMILIES), len(FAMILIES)))
for p in range(len(FAMILIES)):
    rows_p = has_prev & (lag1 == p)
    denom = rows_p.sum()
    if denom == 0:
        continue
    for q in range(len(FAMILIES)):
        T[p, q] = np.sum(rows_p & (nxt == q)) / denom

fig, ax = new_fig(figsize=(6.6, 5.6))
im = ax.imshow(T, aspect="auto", cmap="viridis", vmin=0.0, vmax=float(np.nanmax(T)))
ax.set_xticks(range(len(FAMILIES))); ax.set_xticklabels(FAMILIES)
ax.set_yticks(range(len(FAMILIES))); ax.set_yticklabels(FAMILIES)
ax.set_xlabel("next family (thrown)")
ax.set_ylabel("previous family (in the PA)")
ax.set_title("The bigram the grammar starts from: P(next | previous)\n(each row sums to 1; the diagonal is the repeat tendency)")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label("P(next | previous)")
plt.show()

*Caption.* Read across a row to see what tends to follow that family. A bright diagonal would
mean pitchers repeat; a dim diagonal (as the null-world habit induces) means they avoid
immediate repeats. Off-diagonal structure is the first-order grammar the **L1** view models;
whether the **O** view can find *more* than this — dependence on the pitch *before* the
previous one — is the whole question of `delta_order_L1` (§5).

### (b) The commonest ordered bigrams — the grammar's frequent phrases

The transition matrix normalises within each previous-family row; the raw **bigram frequency**
tells you which ordered two-word phrases actually dominate the corpus. The bars below count the
most common ordered `(previous → next)` family pairs across all sequence-eligible pitches. These
are the phrases the grammar has the most support to model — and, on real data, the ones whose
motif "rules" (§7) will be best estimated.

In [ ]:
pair_codes = lag1[has_prev] * len(FAMILIES) + nxt[has_prev]
vals, counts = np.unique(pair_codes, return_counts=True)
order = np.argsort(counts)[::-1][:15]
labels = [f"{FAMILIES[v // len(FAMILIES)]}→{FAMILIES[v % len(FAMILIES)]}" for v in vals[order]]
heights = counts[order]

fig, ax = new_fig(figsize=(8.4, 4.4))
xs = np.arange(len(order))
ax.bar(xs, heights, color=OKABE_ITO["sky_blue"], edgecolor="white")
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_xlabel("ordered family bigram (previous → next)")
ax.set_ylabel("count in training PAs")
ax.set_title("The grammar's frequent phrases: commonest ordered bigrams\n(support here is what makes a first-order rule estimable)")
plt.show()

*Caption.* The tallest bars are the ordered pairs with the most training support. Same-family
bigrams (e.g. `FF→FF`) appearing prominently is the repeat behaviour the diagonal showed;
cross-family pairs are setup/putaway phrases. Longer ordered phrases (trigrams, the **O**
view's territory) are rarer by construction — every added word multiplies the number of
possible phrases and thins the support, which is why the grammar must *back off* rather than
count deep contexts directly (§3).

### (c) Same-family run lengths — where repeat-suppression lives

A pitcher who throws the same family several times in a row is producing a *run*. The
distribution of **run lengths** (consecutive same-family pitches within a PA) is the direct
fingerprint of repeat behaviour — and the null world's planted "no-three-in-a-row" habit lives
exactly here, as a deficit of runs of length ≥ 3. This is the pattern the top grammar motif
(§7) will report as repeat-*suppression*.

In [ ]:
d = train[["pa_id", "pitch_number", "family"]].sort_values(["pa_id", "pitch_number"])
fam_seq = d["family"].astype("object").to_numpy()
pa_seq = d["pa_id"].to_numpy()
new_run = np.ones(len(fam_seq), dtype=bool)
if len(fam_seq) > 1:
    new_run[1:] = (pa_seq[1:] != pa_seq[:-1]) | (fam_seq[1:] != fam_seq[:-1])
run_id = np.cumsum(new_run)
_, run_lengths = np.unique(run_id, return_counts=True)

max_len = int(run_lengths.max())
bins = np.arange(1, min(max_len, 6) + 2) - 0.5
fig, ax = new_fig()
ax.hist(np.clip(run_lengths, 1, min(max_len, 6)), bins=bins,
        color=OKABE_ITO["bluish_green"], edgecolor="white")
ax.set_xlabel("same-family run length (consecutive pitches, capped at 6)")
ax.set_ylabel("number of runs")
ax.set_title("Where repeat-suppression lives: same-family run lengths\n(a thin tail at length >= 3 is the planted no-three-in-a-row habit)")
frac_ge3 = float(np.mean(run_lengths >= 3))
ax.text(0.97, 0.9, f"P(run length >= 3) = {frac_ge3:.3f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=10, color=OKABE_ITO["vermillion"])
plt.show()

*Caption.* Most runs have length 1 (pitchers mix), and the mass at length ≥ 2 is ordinary
repetition. The interesting quantity is the **deficit** at length ≥ 3: if a pitcher who has
thrown two of a family avoids a third more than chance, runs of 3+ are rarer than a
memoryless model predicts — an **order-2** effect, invisible to L1 (which sees only the single
previous pitch). That deficit is precisely what the grammar's depth-2 level is built to catch,
and what the top motif reports as `X X → P(X)` dropping.

### (d) Tokens per plate appearance — how long the sentences are

Order can only matter in sentences long enough to *have* an order. A PA that ends on pitch 1
carries no within-PA history; the **O** view can only differ from **L1** once a PA reaches
`pitch_number ≥ 3` (it needs two priors to condition on the pitch-before-last). The histogram
shows how much of the corpus is even eligible for higher-order structure.

In [ ]:
pa_len = train.groupby("pa_id")["pitch_number"].max().to_numpy()
fig, ax = new_fig()
bins = np.arange(1, int(pa_len.max()) + 2) - 0.5
ax.hist(pa_len, bins=bins, color=OKABE_ITO["sky_blue"], edgecolor="white")
ax.set_xlabel("pitches in the plate appearance (sentence length)")
ax.set_ylabel("number of plate appearances")
med = float(np.median(pa_len))
ax.axvline(med, color=OKABE_ITO["vermillion"], lw=2)
ax.text(med + 0.1, ax.get_ylim()[1] * 0.9, f"median = {med:.0f}", color=OKABE_ITO["vermillion"])
frac_ge3 = float(np.mean(pa_len >= 3))
ax.set_title(f"How long the sentences are: pitches per PA\n(O can exceed L1 only at length >= 3 — that is {frac_ge3:.0%} of PAs here)")
plt.show()

*Caption.* The mass at length 1–2 is PAs where **O** cannot differ from **L1** — no
pitch-before-last to condition on. The longer the tail, the more room higher-order grammar has
to matter, and the more the `long_pa` (`pitch_number ≥ 3`) and deeper-count slices carry the
signal. Keep this shape in mind reading the by-pitch-number `B_seq` breakdown (§6): bits can
only appear where the sentences are long enough to hold order.

## 3. The model — a two-axis hierarchical Dirichlet grammar (exposition)

No code in this section — just the exact factorization the grammar rests on, in symbols
matched to `model.py`. Full step-by-step derivations are in `THEORY.md`.

### The idea in one sentence

Fuse the two ideas of the study: the **Markov** idea (condition the next token on an ordered
string of previous tokens) and the **Bayesian-pooling** idea (shrink a thin, specific estimate
toward a well-estimated coarser one). The result is a *variable-order* grammar: it uses as much
ordered context as the data can support, and **backs off** to shorter context — ultimately to
the count-and-pitcher base rate — exactly where support runs out. The depth of context used is
not a hyperparameter; it is **earned**.

### Two axes (decision D29)

The row prediction is a **product of experts** over two Dirichlet-multinomial backoff chains.

**Cell axis — the base rate `b`.** This is WS1's C-ladder: the global family marginal
`π_G`, shrunk into a count × handedness cell `ch = (balls, strikes, stand, p_throws)`, then a
further pitcher level:

$$P(a \mid ch) = \frac{n_{ch} + \beta_{ch}\,\pi_G}{N_{ch} + \beta_{ch}},
\qquad
b(a) \equiv P(a \mid ch, \mathrm{pit}) =
     \frac{n_{ch,\mathrm{pit}} + \beta_{\mathrm{pit}}\,P(a\mid ch)}{N_{ch,\mathrm{pit}} + \beta_{\mathrm{pit}}}.$$

The base `b` carries the **count-conditioned repertoire** of the specific pitcher.

**Depth axis — the ordered lift `g_k / m`.** The grammar chain is keyed by the **pitcher**
(pooled over counts, so deep ordered contexts keep support and the lift stays *pitcher-clean*).
Write `m(a) = P(a \mid \mathrm{pit})` for the pitcher marginal (itself shrunk toward `π_G`).
For a context of depth `k`, written chronologically `ctx_k = (w_{t-k}, …, w_{t-1})`, the
depth-`k` distribution is shrunk toward its **suffix** parent:

$$g_k(a \mid ctx_k, \mathrm{pit}) =
     \frac{n + \alpha_k\, g_{k-1}(a \mid ctx_{k-1}, \mathrm{pit})}{N + \alpha_k},
\qquad g_0 \equiv m, \quad ctx_{k-1} = \operatorname{suffix}(ctx_k),$$

where `suffix` drops the **oldest** family `w_{t-k}` — exact suffix-chain backoff. A row is
eligible at depth `k` only when the PA supplies `k` real priors (`pitch_number ≥ k + 1`), so a
context never contains a "no-prior" sentinel.

**Product-of-experts combination.** The base supplies the count × pitcher repertoire; the
grammar supplies the ordered **lift** — how the ordered context reshapes the pitcher's own mix:

$$P(a \mid ch, \mathrm{pit}, ctx_k) \;\propto\; b(a)\,\frac{g_k(a \mid ctx_k, \mathrm{pit})}{m(a)}
\;=\; \underbrace{P(a\mid ch,\mathrm{pit})}_{\text{count} \times \text{pitcher base}}
      \underbrace{\frac{P(a\mid \mathrm{pit}, ctx_k)}{P(a\mid \mathrm{pit})}}_{\text{within-pitcher order lift}},$$

renormalised over the 8 families. When the context is uninformative (`g_k = m`) the prediction
is exactly the base `b`. So the **C** view *is* `b`; **L1** applies the depth-1 lift and **O**
the depth-≤`K_MAX` lift on top of the same base. An unseen context makes `g_k` back off to its
longest observed suffix; an unseen pitcher makes `b` and `m` back off to the count × hand and
global marginals.

**Why key the lift on the pitcher (the crux of D29).** Pooling the grammar over pitchers (a
count-only key) would let the previous family stand in for pitcher identity — `prev = FC`
signals a *cutter pitcher* — so the lift would double-count the repertoire `b` already encodes,
and *hurt*. Nesting the grammar under both count and pitcher removes that confound but
fragments deep contexts until the ordered signal is un-earnable. Conditioning the lift on the
pitcher while pooling over the count keeps both a count × pitcher base *and* a supported,
confound-free ordered lift; it trades on the order effect being largely count-independent (the
count main effect is carried by `b`) — a documented approximation.

### Fitting each concentration, and what a fitted α means

Each level's concentration is fitted **conditional on the posterior means of its parents** (a
top-down pass), pooling the exchangeable cells at that level, by **maximum marginal likelihood**
of the Dirichlet-multinomial (Pólya) evidence — a smooth 1-D optimisation on `log10 α` via
`scipy.optimize`; a method-of-moments (Pearson-overdispersion) estimator is the documented
fallback when a level is too sparse to identify the concentration.

A fitted concentration is a **readout**, not a nuisance:

- **`α_k → ∞` means depth `k` adds nothing.** The depth-`k` contexts look like independent
  draws from their suffix parent, so the fit pools them all the way back and the ordered lift
  at that depth vanishes. The ceiling is effectively unbounded (`10^6`): selection from flat
  contexts genuinely gains little from deep history, so a depth with no signal is pooled to its
  parent — the honest variable-order result. *You will see this literally happen at depths 3–4
  on the null world, whose planted habit is exactly order 2.*
- **A moderate `α_k` means depth `k` carries real ordered signal** — the context's own counts
  win over the backoff prior.

This is why the **effective order** (§7) is a *readout, not a parameter*: a row's earned depth
is the deepest observed context whose own-count weight `ω_k = N_k / (N_k + α_k)` clears a
threshold `τ` (default 0.5). A depth pooled to its ceiling has `ω_k → 0` and never qualifies.

### Lineage — the same machinery, pointed at pitches

The grammar is a hierarchical Bayesian *language model* whose vocabulary is 8 pitch families
instead of words. Its ancestry is the smoothing-and-backoff literature of statistical NLP:
**Katz** (1987) backoff (fall back to a shorter context when the longer one is unseen);
**Kneser–Ney** (1995) smoothing and **Chen & Goodman's** (1999) systematic comparison
(interpolate the higher-order estimate with the lower-order one); the variable-order
**probabilistic suffix trees** of **Ron, Singer & Tishby** (1996) and the prediction study of
**Begleiter, El-Yaniv & Yona** (2004); and — closest of all — the *hierarchical Bayesian*
language models of **MacKay & Peto** (1995), whose hierarchical Dirichlet is exactly the
suffix-chain shrinkage above, and **Teh** (2006), whose hierarchical Pitman–Yor process is the
state-of-the-art refinement. WS2 is that machinery with the words replaced by pitches and the
depth earned by fitted concentrations rather than fixed by hand.

## 4. Fitting the grammar — C, L1, O

Now we fit. For each view we build the two-axis grammar on `train`, predict the next-family
distribution on `val`, and keep both the fitted model (for its diagnostics) and a
standard-schema prediction table (for the shared harness). Each view is fit **once** here and
reused throughout. Recall the view → depth map: **C** = order 0 (base only), **L1** = order 1,
**O** = variable order ≤ `K_MAX`.

In [ ]:
def selection_prediction_df(model, val_table, view):
    proba = model.predict_proba(val_table)
    df = pd.DataFrame({"row_id": val_table["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        df[col] = proba[:, j]
    df["model_id"] = "ws2_bayes_markov"
    df["state_view"] = view
    df["seconds"] = 0.0
    df["peak_mem_mb"] = 0.0
    df["n_params"] = int(model.n_params)
    return df

models = {}
proba_by_view = {}
sel_pred = {}
for v in VIEWS:
    m = vom.fit(train, v, k_max=K_MAX)
    models[v] = m
    proba_by_view[v] = m.predict_proba(val)
    sel_pred[v] = selection_prediction_df(m, val, v)

# Confirm the not-applicable views report honestly rather than fitting (D29).
for na in ("U", "OM"):
    note = vom.not_applicable_note(na)
    print(f"{na}: applicable={note['applicable']}  (not fitted — see message)")
print("fitted grammar views:", list(models))

### Fitted per-depth concentrations, level by level

This is the readout promised in §3. For each view we print the fitted concentration at every
level and the estimator that produced it (`mml` = maximum marginal likelihood, `mom` =
method-of-moments fallback, `default` = too few cells to identify). The cell-axis levels
(`cell`, `pitcher`) carry the base repertoire; the depth-axis levels (`depth1 … depth4`) carry
the ordered lift. **Watch the depth ladder:** a depth whose `α` sits near the `1e6` ceiling is
the model telling you that ordered level adds essentially nothing beyond its suffix.

In [ ]:
rows = []
for v in VIEWS:
    m = models[v]
    conc, meth, ncell = m.concentrations_, m.concentration_methods_, m.n_cells_
    for level in conc:
        rows.append({"view": v, "level": level, "alpha": conc[level],
                     "method": meth[level], "n_cells": ncell[level]})
conc_table = pd.DataFrame(rows)
with pd.option_context("display.float_format", lambda x: f"{x:,.3g}"):
    print(conc_table.to_string(index=False))

**How to read it.** Ignore the `global` root (a weak default prior). The informative cell-axis
rows are `cell` and `pitcher`; the informative depth-axis rows are `depth1 … depth4`. On the
null world the signature is diagnostic: `depth1` and `depth2` fit **moderate** concentrations
(their own counts win — the planted order-2 habit is real), while `depth3` and `depth4` run to
the `1e6` ceiling (no order-3/4 signal exists, so the fit pools them away). That is empirical
Bayes reading the *true generative order* straight off the data: it stops finding structure
exactly where the simulator stopped planting it. On real data, watch how far down the ladder
moderate concentrations persist — that is the depth the data can actually support.

### Selection log loss versus the four count-based references

We score every view through the **shared harness** — no bespoke scoring — and line them up
against the four count-based references (`eval/baselines.py`): global family frequency by
count × hand, pitcher × count, first-order transition, and pitcher × count × prev-family. WS2
is the *serious* hierarchical grammar, so **L1** and **O** should sit at or below the
`pitcher_count_prev` reference. Log loss is the primary selection metric (lower is better).

In [ ]:
sel_reports = {
    v: evaluate_predictions(sel_pred[v], val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    for v in VIEWS
}
sel_loss = {v: sel_reports[v]["slices"]["all"]["action_prob"]["log_loss"] for v in VIEWS}

baseline_loss = {}
for name, mdl in build_baselines(alpha=8.0).items():
    mdl.fit(train)
    proba = mdl.predict_proba(val)
    bdf = pd.DataFrame({"row_id": val["row_id"].to_numpy()})
    for j, col in enumerate(ACTION_PROB_COLS):
        bdf[col] = proba[:, j]
    bdf["model_id"] = f"baseline_{name}"
    bdf["state_view"] = "NA"
    bdf["seconds"] = 0.0
    bdf["peak_mem_mb"] = 0.0
    bdf["n_params"] = 0
    brep = evaluate_predictions(bdf, val, config=CONFIG, n_boot=N_BOOT, seed=SEED)
    baseline_loss[name] = brep["slices"]["all"]["action_prob"]["log_loss"]

# view -> its natural count-based reference (matches run_ws2._REFERENCE_FOR).
REFERENCE_FOR = {"C": "pitcher_count", "L1": "pitcher_count_prev", "O": "pitcher_count_prev"}
print("WS2 selection log loss (val):")
for v in VIEWS:
    ref = baseline_loss[REFERENCE_FOR[v]]
    mark = "<= ref" if sel_loss[v] <= ref + 1e-9 else "> ref"
    print(f"  {v:<3}: {sel_loss[v]:.4f}   ref[{REFERENCE_FOR[v]}]={ref:.4f}  ({mark})")
print("count-based references:")
for k, x in baseline_loss.items():
    print(f"  {k:<20}: {x:.4f}")

In [ ]:
fig, ax = new_fig(figsize=(7.6, 4.4))
xs = np.arange(len(VIEWS))
ax.bar(xs, [sel_loss[v] for v in VIEWS], color=[VIEW_COLORS[v] for v in VIEWS], width=0.6)
for x, v in zip(xs, VIEWS):
    ax.text(x, sel_loss[v], f"{sel_loss[v]:.3f}", ha="center", va="bottom", fontsize=10)

ref_styles = {
    "global_count_hand": (":", "global x count x hand"),
    "pitcher_count": ("--", "pitcher x count"),
    "transition": ("-.", "prev-family x count"),
    "pitcher_count_prev": ("-", "pitcher x count x prev"),
}
for name, (ls, label) in ref_styles.items():
    if name in baseline_loss:
        y = baseline_loss[name]
        ax.axhline(y, color=REF_COLOR, ls=ls, lw=1.2, alpha=0.8)
        ax.text(len(VIEWS) - 0.4, y, f"  {label}", va="center", ha="left", fontsize=8, color=REF_COLOR)

ax.set_xticks(xs); ax.set_xticklabels(VIEWS)
ax.set_xlabel("state view")
ax.set_ylabel("selection log loss (val, lower better)")
ax.set_title("The grammar vs the quick count references\n(bars = WS2 views; black lines = count-based baselines)")
ax.set_xlim(-0.6, len(VIEWS) + 0.9)
plt.show()

*Caption.* Bars are WS2's views; black lines are the reference baselines. **L1** and **O**
should sit **at or below** the `pitcher × count × prev` line — if the serious grammar lost to
the quick baseline, something is wrong. **O** dropping below **L1** is the ordered edge made
visual (quantified with a CI in §5). On the synthetic worlds `stand / p_throws` carry no
signal, so **C** can sit a hair (~0.006 log loss) *above* the `pitcher × count` line — the same
documented, world-specific artifact WS1 reports, expected to reverse on real data where
handedness matters. **O** beating all four references is the headline: the grammar is then the
best next-pitch selector on the board.

### Is the ordered (O) view calibrated?

A good probabilistic model is **calibrated**: when it says "70% four-seam", four-seam should
follow about 70% of the time. The reliability curve bins the O grammar's top-class confidence
and plots, per bin, mean predicted confidence against the observed hit rate. Points on the
diagonal are perfectly calibrated; sagging below it is over-confidence. The grammar's fitted
concentrations exist precisely to keep it honest — a deep context it has barely seen is shrunk
toward its suffix rather than trusted.

In [ ]:
proba_O = proba_by_view["O"]
conf = proba_O.max(axis=1)
pred_fam = np.array(FAMILIES)[proba_O.argmax(axis=1)]
correct = (pred_fam == val["family"].astype("object").to_numpy()).astype(float)
rel = reliability_table(correct, conf, n_bins=10)

fig, ax = new_fig(figsize=(5.4, 5.2))
ax.plot([0, 1], [0, 1], ls="--", color=OKABE_ITO["black"], lw=1, label="perfect calibration")
ax.plot(rel["mean_pred"], rel["frac_pos"], "-o", color=VIEW_COLORS["O"], lw=2)
for _, r in rel.iterrows():
    ax.annotate(f"n={int(r['count'])}", (r["mean_pred"], r["frac_pos"]),
                textcoords="offset points", xytext=(4, -9), fontsize=7, color="gray")
ax.set_xlabel("mean predicted confidence (top class)")
ax.set_ylabel("observed hit rate")
ax.set_title("Is the O grammar calibrated?\n(points on the dashed line = confidence matches reality)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=9, loc="upper left")
plt.show()

*Caption.* Bin counts are annotated so you can discount sparse bins. Systematic sag below the
diagonal at high confidence is over-confidence — a grammar certain about a deep context it has
barely seen. Because the fitted `α_k` shrinks unsupported depths back to their suffix, the
curve should hug the diagonal; a large deviation on real data would point at a depth whose
concentration was under-estimated. Cross-reference with the effective-order distribution (§7):
well-calibrated confidence and modest effective order are the same honesty seen two ways.

## 5. The ordered-selection edge — `delta_order_L1` with a clustered CI

The headline ordered-selection comparison for the grammar is

$$\texttt{delta\_order\_L1} = \text{Loss}(L1) - \text{Loss}(O),$$

the direct out-of-sample selection-log-loss improvement of the full variable-order grammar over
the first-order (previous-pitch) grammar, with a confidence interval bootstrapped over
**pitcher-game clusters** (SPEC §7). A positive value means order *beyond the previous pitch*
sharpens next-pitch prediction.

### Why this is read directly — the contrast with D21

The SPEC §6 ablation is `Δ_order = Loss(min[U, L1]) − Loss(O)`. Decision **D21** flags it as
**negatively biased under the null**, because `min[U, L1]` is the smaller of two noisy losses
and the minimum of noisy quantities is optimistically low (a Jensen gap). WS2 has **no U view**
(a Markov grammar has no unordered multiset context), so it never forms that minimum. Its edge
is `Loss(L1) − Loss(O)` — a straight difference of two losses, **no `min`, so no optimism
bias** — and its clustered CI is therefore read **directly**:

- a CI lower bound **> 0** is genuine ordered selection structure beyond the previous pitch;
- a value **≈ 0** (or slightly negative) reads as *no ordered selection edge beyond L1* —
  **never** as "order hurts" (an out-of-sample predictor free to ignore order cannot be hurt by
  it; and the residual bias here, if any, is mildly *conservative* — O carries more free
  parameters, so under the null its extra flexibility slightly *raises* its loss).

The next cell computes the edge exactly as `run_ws2` does (`log_loss_per_row` for L1 and O,
differenced, with a `clustered_ci`).

In [ ]:
LABELS = list(FAMILIES)
y_val = val["family"].astype("object").to_numpy()
loss_l1 = log_loss_per_row(y_val, proba_by_view["L1"], LABELS)
loss_o = log_loss_per_row(y_val, proba_by_view["O"], LABELS)

clusters = val[["pitcher", "game_pk"]].reset_index(drop=True)

def _edge_metric(pos_idx):
    return float(loss_l1[pos_idx].mean() - loss_o[pos_idx].mean())

edge_ci = clustered_ci(_edge_metric, clusters, n_boot=N_BOOT, seed=SEED)
delta_order_L1 = float(loss_l1.mean() - loss_o.mean())
do_significant = bool(edge_ci["lo"] > 0)

print(f"Loss(L1)          = {loss_l1.mean():.5f}")
print(f"Loss(O)           = {loss_o.mean():.5f}")
print(f"delta_order_L1    = Loss(L1) - Loss(O) = {delta_order_L1:+.5f}")
print(f"95% clustered CI  = [{edge_ci['lo']:+.5f}, {edge_ci['hi']:+.5f}]   (n_boot={edge_ci['n_boot']})")
print(f"significantly positive (CI lower bound > 0)? {do_significant}")

**How to read it.** A CI lower bound above 0 is the clean ordered-selection claim: the full
grammar beats the bigram by more than pitcher-game sampling noise. On the null-world positive
control this is significant *by construction* — the planted order-2 habit is real ordered
structure, and the grammar's committed validation puts the edge at **+0.0063, CI
[+0.0041, +0.0080]** (significant). SPEC §13's honest expectation for *real* data is a
**small** edge — full order adding little beyond the previous pitch — so do not be surprised by
a modest number; be surprised by a large one, and then check §7's support diagnostics before
believing it.

## 6. Bits of predictability — `B_seq` (SPEC §10)

The grammar's second headline read-out is **predictability in bits** (SPEC §10, decision D31):
how many extra **bits** about the next pitch does ordered history supply beyond non-sequence
context? For the ordered predictor `q_O` and the context-only predictor `q_C`, scored on the
*realized* next family `A_t`,

$$B_{\text{seq}} = \frac{1}{n}\sum_t \log_2 \frac{q_O(A_t \mid S_t)}{q_C(A_t \mid X_t)}.$$

A positive value means ordered history makes the actual next pitch **more forecastable** from
pre-release information. Equivalently (see `THEORY.md` §7), `B_seq` is the C-minus-O
cross-entropy difference expressed in bits — the same quantity as the log-loss gap between the
**C** and **O** views, divided by `ln 2`.

> **The SPEC §10 caveat, kept front and centre.** A positive `B_seq` is **finding #1**
> (forecastable ordering / selection structure). It is **not** proof that the ordering helps
> the *batter's outcome* (finding #2), and certainly not that *changing* the sequence would
> help (finding #3). It says the next pitch is more predictable from the sequence — a
> batter-side *information* quantity — nothing more.

In [ ]:
bits = bits_of_predictability(proba_by_view["O"], proba_by_view["C"], y_val)

# Count bucket (pitcher's view), matching run_ws2._count_bucket.
balls = pd.to_numeric(val["balls"], errors="coerce").fillna(0).to_numpy()
strikes = pd.to_numeric(val["strikes"], errors="coerce").fillna(0).to_numpy()
count_bucket = np.full(len(val), "even", dtype=object)
count_bucket[strikes > balls] = "ahead"
count_bucket[balls > strikes] = "behind"
two_strike = strikes >= 2
three_ball = balls >= 3

work = pd.DataFrame({
    "pitch_number": pd.to_numeric(val["pitch_number"], errors="coerce").fillna(0).astype(int).to_numpy(),
    "count_bucket": count_bucket,
})
by_pn = bits_summary(work, bits, by=["pitch_number"])
by_cb = bits_summary(work, bits, by=["count_bucket"])

B_seq_overall = float(bits.mean())
print(f"B_seq overall      = {B_seq_overall:+.4f} bits/pitch")
print(f"  two-strike       = {float(bits[two_strike].mean()):+.4f}" if two_strike.any() else "  two-strike: n/a")
print(f"  three-ball       = {float(bits[three_ball].mean()):+.4f}" if three_ball.any() else "  three-ball: n/a")
print("  by count-bucket  : " + "  ".join(f"{r.count_bucket}={r.mean_bits:+.4f}" for r in by_cb.itertuples()))
print("  by pitch-number  : " + "  ".join(f"t{int(r.pitch_number)}={r.mean_bits:+.3f}" for r in by_pn.head(6).itertuples()))

### `B_seq` by pitch number and by count leverage

Order can only carry bits where there is order to carry them (deeper into the PA) and where the
count gives the pitcher room to sequence (leverage counts). The two figures below break `B_seq`
out the way SPEC §10 asks — by pitch-number and by count situation. The expected shape, seen on
the committed validation, is **≈ 0 bits early, rising with PA depth and count leverage**.

In [ ]:
fig, ax = new_fig(figsize=(7.6, 4.2))
pn = by_pn["pitch_number"].to_numpy()
mb = by_pn["mean_bits"].to_numpy()
keep = pn <= 8
ax.bar(pn[keep], mb[keep], color=VIEW_COLORS["O"], edgecolor="white")
ax.axhline(0.0, color=OKABE_ITO["black"], lw=1)
ax.set_xlabel("pitch number within the PA")
ax.set_ylabel("mean B_seq (bits)")
ax.set_title("Bits appear where order can exist: B_seq rises with PA depth\n(near 0 at t1-t2, growing as the sentence lengthens)")
plt.show()

In [ ]:
fig, ax = new_fig(figsize=(7.4, 4.2))
cats = ["ahead", "even", "behind", "two_strike", "three_ball"]
means = {r.count_bucket: r.mean_bits for r in by_cb.itertuples()}
vals_plot = [means.get("ahead", np.nan), means.get("even", np.nan), means.get("behind", np.nan),
             float(bits[two_strike].mean()) if two_strike.any() else np.nan,
             float(bits[three_ball].mean()) if three_ball.any() else np.nan]
colors = [OKABE_ITO["sky_blue"]] * 3 + [OKABE_ITO["vermillion"], OKABE_ITO["orange"]]
xs = np.arange(len(cats))
ax.bar(xs, vals_plot, color=colors, edgecolor="white")
ax.axhline(0.0, color=OKABE_ITO["black"], lw=1)
for x, vv in zip(xs, vals_plot):
    if np.isfinite(vv):
        ax.text(x, vv, f"{vv:+.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels(cats, rotation=15)
ax.set_ylabel("mean B_seq (bits)")
ax.set_title("More bits under count leverage: B_seq by count situation\n(two-strike / three-ball counts give the most room to sequence)")
plt.show()

*Caption.* The by-pitch-number bars should start near zero (`t1`, `t2` have little or no
ordered context) and grow as the PA lengthens; the by-count bars should be largest in
two-strike and three-ball counts, where the pitcher has the most latitude to set up a putaway.
The committed null-world validation shows exactly this: overall **+0.0120 bits**, two-strike
**+0.0264**, three-ball **+0.0466**, and by pitch number `t1=+0.000, t2=−0.001, t3=+0.023,
t4=+0.026, t5=+0.034`. To put **0.0120 bits** in human terms: one full bit is one yes/no
question answered; 0.012 bits is about **one-eightieth** of a yes/no question of next-pitch
information per pitch — real and directional, but small (§10, and `THEORY.md` §7).

## 7. Grammar exhibits — how deep, and which rules

Two exhibits turn the fitted grammar into something you can inspect: the **effective-order
distribution** (how deep the ordered dependence actually reaches) and the **top motifs** (the
grammar's "rules", ranked by how much an ordered context moves the next-token distribution).

### Effective order — the depth the data actually earns

A row's **effective order** is the deepest grammar depth `k` whose context `(pit, ctx_k)` was
seen in training *and* whose own-count weight

$$\omega_k = \frac{N_k}{N_k + \alpha_k}$$

reaches `τ` (default 0.5) — i.e. the context's own data, not the backoff prior, drives its
distribution. A depth with no ordered signal fits `α_k` at the ceiling, so `ω_k` collapses and
the effective order drops to the coarser context. This is a **readout, not a parameter**: it
reports where the grammar's confidence in ordered structure actually lands, row by row.

In [ ]:
eff = models["O"].effective_order(val, tau=vom.EFFECTIVE_ORDER_TAU)
dist = eff["distribution"]
orders = sorted(dist)
fig, ax = new_fig(figsize=(7.2, 4.2))
ax.bar(orders, [dist[k] for k in orders], color=VIEW_COLORS["O"], edgecolor="white")
for k in orders:
    ax.text(k, dist[k], f"{dist[k]:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("effective order (earned depth, tau = %.2f)" % eff["tau"])
ax.set_ylabel("fraction of validation rows")
ax.set_title("How deep the data lets us go: effective-order distribution\n(mean = %.3f, mass at order >= 2 = %.3f)" % (eff["mean"], eff["frac_ge2"]))
plt.show()

*Caption.* Most mass typically sits at order 0–1: deep ordered contexts rarely clear their
support threshold, which is the honest "how much order the data can resolve" statement. A
materially non-zero **order ≥ 2** mass is real, resolvable ordered structure. On the null-world
positive control the distribution is bimodal by design — `k0 = 0.704, k2 = 0.296`, with
`k1 = k3 = k4 = 0` (mean **0.593**): the planted habit is a *pure* order-2 effect, so rows
either earn depth 2 or fall to the base, and nothing earns depth 1, 3, or 4. On real data
expect a smoother tail, thinning with depth.

### The grammar's rules — top motifs by lift

The **motifs** are the ordered contexts whose next-token distribution moves most versus their
**suffix** (the same context with the oldest family dropped). The code ranks every context
(pooled over the pitchers that share the ordered family context) by the Kullback–Leibler
divergence

$$D_{\mathrm{KL}}(p_{\text{ctx}} \,\|\, p_{\text{suffix}})
   = \sum_a p_{\text{ctx}}(a)\,\log_2 \frac{p_{\text{ctx}}(a)}{p_{\text{suffix}}(a)},$$

and reports, for the top contexts, the single family whose probability moved most — its
before/after probability, the `log2` ratio, and the direction (`suppress` if the probability
drops, `promote` if it rises). The rows below print in exactly the headline format.

In [ ]:
motifs = models["O"].top_motifs(min_support=30, top_n=20)
print(f"{len(motifs)} motifs at support >= 30; top 12 by KL(ctx || suffix):\n")
print(f"  {'context':<14} {'depth':>5} {'support':>8} {'move':>22} {'dir':>9} {'KL':>7}")
print("  " + "-" * 70)
for m in motifs[:12]:
    ctx = " ".join(m["context"])
    move = f"P({m['top_family']}) {m['p_suffix']:.2f}->{m['p_ctx']:.2f}"
    print(f"  {ctx:<14} {m['depth']:>5} {m['support']:>8} {move:>22} {m['direction']:>9} {m['kl']:>7.3f}")

*Caption / face-validity.* Read each row as a rule: *"after this ordered context, the
probability of this family moves this way."* On the null-world positive control every top motif
is the planted **repeat-suppression** habit — `SI SI → P(SI) 0.35→0.22`, `FS FS → 0.30→0.19`,
`FF FF → 0.31→0.20`, `CU CU → 0.25→0.17`, `FC FC → 0.32→0.24` — a same-family run making that
family *less* likely next, correctly signed, exactly the no-three-in-a-row rule the simulator
planted (that structural check, `depth ≥ 2` same-family run suppressed, is what the D30 detect
verdict keys on).

On **real** data, face-valid motifs would look like recognisable sequencing tendencies:
same-family repeat suppression (pitchers avoid becoming predictable), and **fastball → breaking
setups** (a fastball raising the next-pitch probability of an offspeed/breaking family, or a
two-strike shift toward a putaway pitch). Motifs with no baseball reading, or that ride on
contexts barely clearing the support floor, are the ones to treat as possible overfit — the
Motif branch (§9) makes that call.

## 8. Falsification — the D30 story (detect, then collapse)

A model that merely *runs* has proven nothing (SPEC §0). WS2's acceptance wiring is two
verdicts, and this is where the **D30 inversion** pays off.

**The setup (D20 → D30).** The oracle's worlds were built to separate finding #1 from finding
#2 (D20): the null world forbids an ordered *outcome* effect but **plants an ordered *selection*
habit** (no-three-in-a-row). For an outcome model that world is a true null; for WS2's
*selection* grammar it is a **positive control** — the planted habit is exactly the order-2
structure the grammar should find. So WS2's two checks are:

1. **Detect (positive control).** On the null world the grammar must fire `GRAMMAR_DETECTED`:
   O beats L1 on selection log loss, the effective order ≥ 2 mass is material, **and** a
   repeat-suppression motif is present.
2. **Collapse (negative control).** The genuine negative control is a **stratified history
   permutation** — scramble whole ordered-history vectors among rows sharing a
   `(count × hand cell, pitch_number)` stratum (which fixes the count mix and the PA depth, so
   a permuted context has the right length but no genuine ordered dependence). Refit and
   re-evaluate on the permuted contexts; the edge and the effective order must collapse toward
   the base. Verdict `COLLAPSES_UNDER_PERMUTATION`.

The cell below runs the whole WS2 pipeline on the null world (fast, no data files) and prints
both verdicts. It uses `seed=7`, the committed-validation seed, so the numbers reproduce the
values quoted throughout.

In [ ]:
falsify = run_ws2(synth="null", n_games=300, seed=7, n_boot=50, write_outputs=False)
v = falsify["verdicts"]
do = falsify["delta_order_L1"]
eo = falsify["grammar_exhibits"]["effective_order"]
perm = falsify["permutation_control"]

print("=" * 70)
print(" WS2 D30 acceptance — null world (WS2's POSITIVE control)")
print("=" * 70)
print(f" DETECT  : {v['grammar_detected_verdict']}")
print(f"   O beats L1            : {v['detected_components']['o_beats_l1']}  "
      f"(delta_order_L1 = {do['point']:+.4f} CI[{do['ci']['lo']:+.4f}, {do['ci']['hi']:+.4f}])")
print(f"   order >= 2 mass       : {v['detected_components']['order_ge2_mass']:.3f}  (material > 0.03)")
print(f"   repeat-suppression    : {v['detected_components']['repeat_suppression_motif']}")
print(f" CONTROL : {v['collapse_verdict']}")
cc = v["collapse_components"]
print(f"   ordered edge          : {cc['real_edge']:+.4f}  ->  {cc['perm_edge']:+.4f}  (under permutation)")
print(f"   order >= 2 mass        : {eo['frac_ge2']:.3f}  ->  {cc['perm_order_ge2_mass']:.3f}")
print("=" * 70)

**Completed synthetic validation (state these as done).** The committed WS2 run on the null
world produces, deterministically:

- **`GRAMMAR_DETECTED`** — O = **1.3322** < L1 = **1.3385** (delta_order_L1 = **+0.0063**, CI
  **[+0.0041, +0.0080]**, significant); effective order ≥ 2 mass = **0.296**; all top-5 motifs
  the planted repeat-suppression habit, correctly signed.
- **`COLLAPSES_UNDER_PERMUTATION`** — the ordered edge falls **+0.0063 → +0.0001** and the
  order ≥ 2 mass falls **0.296 → 0.000** once histories are permuted within
  `(count × hand, pitch_number)` strata.
- **`B_seq`** = **+0.0120 bits** overall, **0** at `t ≤ 2`, rising with PA depth and count
  leverage (two-strike **+0.0264**, three-ball **+0.0466**) — textbook.

Both halves matter. Detection alone could be a pitcher-identity artifact; the collapse proves
the edge is *ordered dependence*, not the base rate in disguise. The reference tables (`C`
losing ~0.006 to `pitcher_count` on the synthetic world because handedness is uninformative
there) are the same documented artifact WS1 reports, expected to reverse on real data.

*(The oracle's `'synth_positive'` world adds a planted ordered **outcome** effect on whiffs.
That is a finding-#2 fixture and is **irrelevant to a selection grammar** — WS2 neither needs
nor claims it; it is available via `DATA_MODE='synth_positive'` for completeness only.)*

## 9. Results — branched interpretation

This is the section that turns numbers into a verdict. The code cell inspects the quantities
computed above and prints **which branch applies** on each of three axes; the markdown that
follows holds the pre-written interpretation for every branch, so the conclusion is ready the
instant the cells finish.

- **Grammar axis** (does order help *selection*, and how deep?): **G1 / G2 / G3**.
- **Bits axis** (is the ordering forecastable?): **B1 / B2**.
- **Motif axis** (do the rules make baseball sense?): **M1 / M2** (qualitative).

The thresholds below are documented, not tuned: `SEL_MATERIAL` (a small next-pitch log-loss
gain that counts as "beats C") and `BITS_MATERIAL` (a small `B_seq` that counts as
"forecastable"). The Grammar axis leans on `delta_order_L1`'s CI, read directly under the §5
logic.

In [ ]:
# --- Grammar axis: G1 (order beyond L1) / G2 (first-order adequate) / G3 (no history) ---
SEL_MATERIAL = 3e-3   # log-loss (nats): a small-but-real next-pitch sharpening vs C
gain_L1_over_C = sel_loss["C"] - sel_loss["L1"]
gain_O_over_C = sel_loss["C"] - sel_loss["O"]
best_hist_gain = max(gain_L1_over_C, gain_O_over_C)

if best_hist_gain <= SEL_MATERIAL:
    grammar_branch = "G3"            # nothing beats C
elif do_significant:                 # O beats L1 by more than the clustered CI
    grammar_branch = "G1"
else:
    grammar_branch = "G2"            # history beats C, but O ~ L1

# --- Bits axis: B1 (forecastable) / B2 (not) ---
BITS_MATERIAL = 5e-3   # bits/pitch
bits_branch = "B1" if B_seq_overall > BITS_MATERIAL else "B2"

# --- Motif axis (qualitative heuristic): repeat-suppression or supported cross-family rules ---
def _is_repeat_suppression(m):
    return m["depth"] >= 2 and len(set(m["context"])) == 1 and m["direction"] == "suppress" \
        and m["top_family"] == m["context"][-1]
top_motifs_shown = motifs[:10]
n_repeat_supp = sum(_is_repeat_suppression(m) for m in top_motifs_shown)
n_supported = sum(m["support"] >= 30 for m in top_motifs_shown)
motif_face_valid = (n_repeat_supp >= 1) and (n_supported == len(top_motifs_shown))
motif_branch = "M1" if motif_face_valid else "M2"

print("=" * 68)
print(" WS2 RESULTS — branch selector")
print("=" * 68)
print(f" world / mode          : {truth.get('world', DATA_MODE)}")
print(f" C / L1 / O log loss   : {sel_loss['C']:.4f} / {sel_loss['L1']:.4f} / {sel_loss['O']:.4f}")
print(f" best history gain vs C: {best_hist_gain:+.4f}   (material {SEL_MATERIAL:+.4f})")
print(f" delta_order_L1        : {delta_order_L1:+.5f}  CI[{edge_ci['lo']:+.5f}, {edge_ci['hi']:+.5f}]")
print(f" B_seq overall         : {B_seq_overall:+.4f} bits   (material {BITS_MATERIAL:+.4f})")
print(f" top-10 motifs         : repeat-suppression={n_repeat_supp}, all-supported={n_supported == len(top_motifs_shown)}")
print("-" * 68)
print(f" GRAMMAR BRANCH        : {grammar_branch}")
print(f" BITS BRANCH           : {bits_branch}")
print(f" MOTIF BRANCH          : {motif_branch}")
print("=" * 68)
print(" -> read the matching branch write-ups in the markdown below.")

### Grammar branch G1 — *ordered selection grammar beyond the previous pitch*

`delta_order_L1` clears zero: the full variable-order grammar beats the bigram by more than the
clustered CI. **Order carries selection information past the previous pitch** — the strong
version of finding #1 at the grammar rung. Characterise it with the two §7 exhibits: the
**effective-order distribution** says *how deep* (where the ≥ 2 mass sits), and the **motifs**
say *which rules* drive it. For the project this is a live, higher-order selection signal for
WS3 (features) and WS6 (a learned representation) to confirm and extend; it sharpens the
headline question from "does history matter?" to "*ordered* history matters, to depth ~k." It
still says nothing about outcomes — a more predictable pitcher is not yet an exploitable one.

### Grammar branch G2 — *first-order adequate (the grammar is mostly bigrams)*

`delta_order_L1 ≈ 0` (not significantly positive), but **L1** and **O** both beat **C**
materially: within-PA history helps selection, yet the *order* beyond the immediately preceding
pitch adds little. Selection memory is **one pitch deep** — the grammar is essentially a set of
calibrated bigrams. This is SPEC §13's explicitly anticipated result ("O barely beats L1"), and
a clean one: it tells WS3 it need only carry the previous pitch, not the full ordered history,
to capture the selection signal, and it sets the bar WS6's learned representation must clear to
justify going deeper. Report it without embarrassment — a modest ordering effect is a real,
publishable finding.

### Grammar branch G3 — *count/pitcher-driven only*

Neither history view beats **C** by more than `SEL_MATERIAL`: next-pitch selection is driven by
count and pitcher identity, with within-PA order adding essentially nothing. This is
*surprising* against the conventional wisdom that pitchers sequence — so treat it as a claim to
audit, not accept. Check §4's backoff/support diagnostics and §7's effective-order distribution
**first**: if deep ordered contexts simply never cleared their support threshold (all mass at
order 0), the honest reading is "the grammar could not resolve order at this data scale," a
support statement rather than a baseball one, and the case passes to WS3/WS6 with more
resolving power. Only if support is ample and order still adds nothing is G3 a substantive
finding about selection.

### Bits branch B1 — *materially positive `B_seq`*

Ordered history makes the next pitch **more forecastable** from pre-release information; report
the by-slice pattern (more bits deeper in the PA and under count leverage). This is a
batter-side **information leak** — the raw material for SPEC §10's predictability/exploitability
frontier that the capstone (WS7) assembles. State the caveat every time: forecastable ≠
exploitable. Positive `B_seq` is finding #1 in bits; whether those bits translate into outcome
value (finding #2) or prescriptive gain (finding #3) is tested downstream, not here.

### Bits branch B2 — *`B_seq` ≈ 0*

The ordering is **not forecastable** beyond context: knowing the sequence adds no measurable
bits about the next pitch. Consistent with a G2/G3 selection reading, and a clean negative for
the predictability frontier — the sequence, whatever else it does, does not leak next-pitch
information a batter-side model could use.

### Motif branch M1 — *motifs reproduce known patterns (face-valid)*

The top grammar rules read as real sequencing tendencies — repeat suppression, fastball →
breaking-ball setups, two-strike putaway shifts — and rest on adequate support. The grammar is
learning baseball, not noise; the motif table is then a genuine descriptive deliverable, and a
qualitative corroboration of whichever Grammar branch fired.

### Motif branch M2 — *arbitrary motifs (inspect)*

The top rules lack a clear baseball reading, or ride on contexts barely clearing the support
floor. Treat the motif exhibit as provisional: re-inspect support, raise `min_support`, and
cross-check against the effective-order distribution (if order ≥ 2 mass is tiny, the deep
motifs are estimated from little and are the first suspects for overfit).

## 10. Discussion and limitations

**The finding #1 vs finding #2 firewall — stated hard.** WS2 is a **selection** grammar and
*nothing it produces speaks to outcome value.* A significant `delta_order_L1`, a deep effective
order, a positive `B_seq`, a table of face-valid motifs — every one of these is a statement that
*the next pitch is more predictable given the ordered history.* None of them is evidence that
the ordering **helps the pitcher's outcome** (finding #2), still less that **changing** the
sequence would help (finding #3). A perfectly predictable pitcher who nonetheless throws
unhittable pitches loses nothing; a forecastable sequence is an *information* fact, not a *run
value* fact. The outcome question is WS3's (the decomposed outcome stack) and the prescriptive
question is the OPE/RL workstreams' (WS4/5/7). Do not let a striking selection result leak
across that firewall.

**What the grammar can and cannot see.**

- **Family granularity hides within-family variation.** The tokens are 8 pitch *families*, so
  the grammar is blind to velocity, location, and movement *within* a family — a two-seam
  played off a four-seam, a backfoot slider versus a backdoor one. A sequencing effect that
  lives in pitch *physics* rather than pitch *names* is invisible here (the same
  mechanism-blindness WS1 quantified at ~3% recovery of a planted velocity effect). WS3 puts
  those quantities in as features.
- **No cross-PA memory (D29).** The grammar is a *within-PA* chain; matchup memory (earlier PAs
  this game / season / career) is the **OM** view, which a Markov context does not span. WS2
  reports `Δ_matchup = N/A` and hands cross-PA adaptation to the pooled/embedding models.
- **U is not a Markov context (D29).** An order-invariant multiset of prior families is not an
  ordered string; the unordered question is answered by the tables (WS1's U) and the flexible
  models, not by the grammar.
- **`pitch_type` is a classifier output**, not the battery's intent (SPEC §1); a mislabeled
  pitch is a mislabeled token. Family granularity blunts this but cannot remove it.

**Inheritance — what the later rungs do with this.**

- **WS3 (GBDT stack, the centerpiece)** replaces the grammar's discrete cells with **features**,
  so an ordered velocity/location transition becomes an input in its own right rather than
  something proxied through family names — and it adds the **outcome** side the grammar
  deliberately omits. WS2's `delta_order_L1` and `B_seq` are the selection bar WS3's behavior
  model must clear and extend.
- **WS6 (deep sequence, GRU)** asks the sharp question WS2 sets up: does a **learned
  representation** rediscover this *explicit* grammar — the same effective depth, the same
  motifs — or find ordered structure the family-token chain cannot express? If the GRU merely
  reproduces WS2's edge, the grammar was the honest model all along; if it exceeds it, the gap
  measures what family tokens leave on the table.

**The through-line.** WS2's contribution is a *calibrated, honest depth read* on ordered
selection: it spends resolution only where support earns it, reports how deep the data actually
lets it go, and keeps the selection finding rigorously walled off from the outcome finding. That
is the rung the rest of the ladder builds on.

## 11. Reproducibility appendix

### Exact Phase-2 commands (RUNBOOK Step WS2.1)

The real-data run is driven by the WS2 CLI, on the train seasons (2021–2023) with validation on
2024:

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
python workstreams/ws2_bayes_markov/run_ws2.py --table data/processed/decision_table.parquet --out results/ws2/ --views C L1 O
```

The synthetic CI equivalents (no data needed, ~5 s each). The null world is WS2's **positive
control**:

```powershell
python workstreams/ws2_bayes_markov/run_ws2.py --synth null --out results/ws2_null/
python workstreams/ws2_bayes_markov/run_ws2.py --synth positive --out results/ws2_pos/
```

Lower `--n-boot` (default 200) if the clustered-bootstrap CIs are slow on the full validation
season; raise/lower `--k-max` to change the grammar-order cap.

### Output files (under `--out`, gitignored)

- `predictions_real_<view>.parquet` — standard-schema selection predictions per view (C/L1/O).
- `ws2_report_real.json` — the full report: per-view losses, references, `delta_order_L1` + CI,
  `B_seq` slices, effective-order distribution, top motifs, D30 verdicts, fitted α per depth.
- `motifs_real.csv` — the ranked grammar rules (ordered context → next-token lift vs suffix).
- `ws2_real.runmeta.json` — wall-clock seconds, peak RAM, row counts — the raw material for the
  SPEC §7 performance-vs-compute Pareto plot.

### Package versions (this environment)

In [ ]:
import scipy
print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("matplotlib :", matplotlib.__version__)
print("seed       :", SEED)
print("data mode  :", DATA_MODE)
print("K_MAX      :", K_MAX)
print("n_boot     :", N_BOOT)